In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import rioxarray

from pathlib import Path

root = Path.cwd()

#### Get spatial samples for training and testing

In [2]:
def make_block_map(valid_2d, block_size,test_size,ny,nx):

    ## find blocks that are mostly valid pixels
    n_blocks_y = ny // block_size   # drop partial edge tiles
    n_blocks_x = nx // block_size
    n_blocks   = n_blocks_y * n_blocks_x
    print(f'number of blocks: {n_blocks}')
   
    usable = np.zeros((n_blocks_y, n_blocks_x), dtype=bool)
    min_pixels = int(block_size * block_size * 0.10)

    for by in range(n_blocks_y):
        for bx in range(n_blocks_x):
            y0, y1 = by * block_size, (by + 1) * block_size
            x0, x1 = bx * block_size, (bx + 1) * block_size
            
            n_valid = valid_2d[y0:y1, x0:x1].sum()
            usable[by, bx] = n_valid >= min_pixels
               

    usable_indices = list(zip(*np.where(usable)))
    n_usable = len(usable_indices)
    n_test   = max(1, int(round(n_usable * test_size)))
    n_train  = n_usable - n_test

    labels = np.array([0] * n_train + [1] * n_test, dtype=np.uint8)
    rng = np.random.default_rng(42)
    rng.shuffle(labels)

    # assign labels to blocks
    block_map = np.full((ny, nx), 255, dtype=np.uint8)
    for i, (by, bx) in enumerate(usable_indices):
        y0, y1 = by * block_size, (by + 1) * block_size
        x0, x1 = bx * block_size, (bx + 1) * block_size
        block_map[y0:y1, x0:x1] = labels[i]
    
    return block_map

def plot_block_map(block_map):

    cmap = {0: [0.2, 0.5, 0.8], 1: [0.9, 0.3, 0.2], 255: [0.85, 0.85, 0.85]}
    rgb = np.zeros((*block_map.shape, 3), dtype=np.float32)
    for val, colour in cmap.items():
        mask = block_map == val
        rgb[mask] = colour
  
    # Downsample for display if image is large
    step = max(1, block_map.shape[0] // 1000)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(rgb[::step, ::step])
    
    ax.legend(handles=[
        Patch(color=cmap[0], label="Train blocks"),
        Patch(color=cmap[1], label="Test blocks"),
        Patch(color=cmap[255], label="Excluded"),
    ], loc="lower right")
    ax.set_title("Spatial block train/test partition")
    ax.axis("off")
    plt.tight_layout()

    plt.show()




## get pixel indices within each block
def sample_indices_nonstratified(pixels_per_block,valid_2d,block_size,block_map,ny,nx,random_state):
    train_indices = []
    test_indices = []

    rng = np.random.default_rng(random_state)

    n_blocks_y = ny // block_size   # drop partial edge tiles
    n_blocks_x = nx // block_size
   
    for by in range(n_blocks_y):
        for bx in range(n_blocks_x):
            y0, y1 = by * block_size, (by + 1) * block_size
            x0, x1 = bx * block_size, (bx + 1) * block_size

            label = block_map[y0, x0]   # 0=train, 1=test, 255=excl
            if label == 255:
                continue

            block_valid_yx = np.argwhere(valid_2d[y0:y1, x0:x1]) # get valid pixels for this block
            if len(block_valid_yx) == 0:
                continue

            # sample given number of pixels
            n_sample = min(pixels_per_block, len(block_valid_yx))
            chosen   = rng.choice(len(block_valid_yx), size=n_sample, replace=False)
            yx       = block_valid_yx[chosen]

            # flat_idx = row * array_width + col
            flat_idx = (y0 + yx[:, 0]) * nx + (x0 + yx[:, 1])

            if label == 0:
                train_indices.append((flat_idx, by, bx))
            else:
                test_indices.append((flat_idx, by, bx))
    
    return train_indices, test_indices



def get_pixels(index_list,bands_flat,canopy_flat,band_names):
    all_flat  = np.concatenate([t[0] for t in index_list])
    block_rows = np.concatenate([np.full(len(t[0]), t[1]) for t in index_list])
    block_cols = np.concatenate([np.full(len(t[0]), t[2]) for t in index_list])

    sort_order = np.argsort(all_flat)
    all_flat   = all_flat[sort_order]
    block_rows = block_rows[sort_order]
    block_cols = block_cols[sort_order]

    band_vals   = bands_flat[:, all_flat].compute()   # (n_bands, N)
    canopy_vals = canopy_flat[all_flat]    # (N,)


    # filter out nans/infs
    valid = np.all(np.isfinite(band_vals), axis=0) & np.isfinite(canopy_vals)
    band_vals   = band_vals[:, valid]
    canopy_vals = canopy_vals[valid]
    block_rows  = block_rows[valid]
    block_cols  = block_cols[valid]

    df = pd.DataFrame(band_vals.T, columns=band_names)
    df["canopy_pct"]    = canopy_vals
    df["canopy_binary"] = (canopy_vals > 0).astype(int)
    df["block_row"]     = block_rows
    df["block_col"]     = block_cols
    return df

In [3]:
## read in lidar raster and features
output_label = 'nyc18'


lidar_raster = rioxarray.open_rasterio(root/ 'data' / output_label / f'{output_label}_tree_canopy_lidar.tif').squeeze('band',drop=True)
features = xr.open_zarr(root/'data'/output_label/f'{output_label}_seasonalstats.zarr')
features = features['features']

# keep = [b for b in features['band'].values if 'correlation' not in str(b)]
#keep = [b for b in features['band'].values if not any(t in str(b) for t in ['contrast', 'dissimilarity', 'homogeneity',
               # 'ASM', 'variance','correlation', 'entropy'])]
# features = features.sel(band=keep)

# define inputs
n_bands = features.sizes["band"]
ny      = features.sizes["y"]
nx      = features.sizes["x"]
band_names = [f"band_{b}" for b in features.coords["band"].values]

bands_flat = features.data.reshape(n_bands, ny * nx)
canopy_flat = lidar_raster.values.ravel()
canopy_2d = lidar_raster.values

valid_2d = np.isfinite(lidar_raster.values)
block_size = 100

In [9]:
assert features.sizes['x'] == lidar_raster.shape[1], 'lidar and feature rasters have different extents'
assert features.sizes['y'] == lidar_raster.shape[0], 'lidar and feature rasters have different extents'
assert all(features.y.values[:3] == lidar_raster.y.values[:3]), 'lidar and feature raster grids not aligned'
assert all(features.x.values[:3] == lidar_raster.x.values[:3]), 'lidar and feature raster grids not aligned'

In [15]:
print(features.sizes, lidar_raster.shape)          # y,x must match exactly
print(features.dims)                               # must be ('band','y','x'), not transposed
print(features.y.values[:3], lidar_raster.y.values[:3])  # same grid, not just same count
print(features.x.values[:3], lidar_raster.x.values[:3])

Frozen({'band': 210, 'y': 4686, 'x': 4670}) (4686, 4670)
('band', 'y', 'x')
[4529950. 4529940. 4529930.] [4529950. 4529940. 4529930.]
[563070. 563080. 563090.] [563070. 563080. 563090.]


In [ ]:
block_map = make_block_map(valid_2d=valid_2d,block_size=block_size,test_size=0.3,ny=ny,nx=nx)

## save blocks to use later as mask
np.savez(root / 'data' / 'train_test_blocks.npz',block_map)
plot_block_map(block_map=block_map)

In [ ]:
block_map = np.load(root / 'data' / 'train_test_blocks.npz')
block_map = block_map['arr_0']

In [ ]:
#### nonstratified sample for classifier
random_state = 42 

def spatial_sample(output_label,model,random_state):

    train_indices, test_indices = sample_indices_nonstratified(pixels_per_block=100,block_size=block_size,block_map=block_map,valid_2d=valid_2d,ny=ny,nx=nx,random_state=random_state)
    train_df = get_pixels(train_indices,bands_flat=bands_flat,canopy_flat=canopy_flat,band_names=band_names)
    test_df = get_pixels(test_indices,bands_flat=bands_flat,canopy_flat=canopy_flat,band_names=band_names)

    test_df.to_parquet(root / 'data' / output_label / f'{output_label}_test_df_{model}.parquet',compression='zstd')
    train_df.to_parquet(root / 'data' / output_label / f'{output_label}_train_df_{model}.parquet',compression='zstd')

    print(f'{model} train pixels: {train_df.shape[0]}\n{model} test pixels: {test_df.shape[0]}')

train pixels: 53808
test pixels: 22775


In [18]:
random_state = 0

train_indices_rgr, test_indices_rgr = sample_indices_nonstratified(pixels_per_block=100,block_size=block_size,block_map=block_map,valid_2d=valid_2d,ny=ny,nx=nx,random_state=random_state)
train_df_rgr = get_pixels(train_indices_rgr,bands_flat=bands_flat,canopy_flat=canopy_flat,band_names=band_names)
test_df_rgr = get_pixels(test_indices_rgr,bands_flat=bands_flat,canopy_flat=canopy_flat,band_names=band_names)

test_df_rgr.to_parquet(root / 'data' / siteyear / f'{siteyear}_test_df_rgr_nonstratified.parquet',compression='zstd')
train_df_rgr.to_parquet(root / 'data' / siteyear / f'{siteyear}_train_df_rgr_nonstratified.parquet',compression='zstd')

print(f'train pixels: {train_df_rgr.shape[0]}\ntest pixels: {test_df_rgr.shape[0]}')

train pixels: 53897
test pixels: 22751


In [37]:
## which features have the most missing data

all_flat  = np.concatenate([t[0] for t in train_indices_clf])
band_vals   = bands_flat[:, all_flat].compute()
frac = np.isfinite(band_vals).mean(axis=1)
for name, f in sorted(zip(band_names, frac), key=lambda t: t[1])[:30]:
    print(f"{f:.3f}  {name}")

0.692  band_contrast_fall_delta
0.692  band_dissimilarity_fall_delta
0.692  band_homogeneity_fall_delta
0.692  band_ASM_fall_delta
0.692  band_variance_fall_delta
0.692  band_entropy_fall_delta
0.694  band_contrast_spring_delta
0.694  band_dissimilarity_spring_delta
0.694  band_homogeneity_spring_delta
0.694  band_ASM_spring_delta
0.694  band_variance_spring_delta
0.694  band_entropy_spring_delta
0.696  band_contrast_summer
0.696  band_dissimilarity_summer
0.696  band_homogeneity_summer
0.696  band_ASM_summer
0.696  band_variance_summer
0.696  band_entropy_summer
0.698  band_psr_spring_delta
0.698  band_ndr_spring_delta
0.699  band_B02_spring_delta
0.699  band_B03_spring_delta
0.699  band_B04_spring_delta
0.699  band_B05_spring_delta
0.699  band_B06_spring_delta
0.699  band_B07_spring_delta
0.699  band_B08_spring_delta
0.699  band_B8A_spring_delta
0.699  band_B11_spring_delta
0.699  band_B12_spring_delta


#### archive

In [ ]:
def sample_indices_stratified(pixels_per_block,block_size,block_map,valid_2d,canopy_2d,ny,nx,random_state,bins=np.array([0, 1e-6, 10, 20, 30, 40, 50, 60, 70, 80, 90, 101.0])):

    rng = np.random.default_rng(random_state)
    
    train_indices = []
    test_indices = []

    n_blocks_y = ny // block_size   # drop partial edge tiles
    n_blocks_x = nx // block_size   
   
    n_bins = len(bins) - 1
    per_bin = max(1, pixels_per_block // n_bins) # num pixels in each bin

    for by in range(n_blocks_y):
        for bx in range(n_blocks_x):
            y0, y1 = by * block_size, (by + 1) * block_size
            x0, x1 = bx * block_size, (bx + 1) * block_size

            label = block_map[y0, x0]   # 0=train, 1=test, 255=excl
            if label == 255:
                continue

            block_valid_yx = np.argwhere(valid_2d[y0:y1, x0:x1]) # get valid pixels for this block
            if len(block_valid_yx) == 0:
                continue

            # sample test pixels without stratification
            if label==1:
                n_sample = min(pixels_per_block, len(block_valid_yx))
                chosen   = rng.choice(len(block_valid_yx), size=n_sample, replace=False)
                yx       = block_valid_yx[chosen]

                # flat_idx = row * array_width + col
                flat_idx = (y0 + yx[:, 0]) * nx + (x0 + yx[:, 1])

                test_indices.append((flat_idx, by, bx))

            elif label==0:
            # stratified sampling within block
            # canopy values for valid pixels in this block
                yy = y0 + block_valid_yx[:, 0]
                xx = x0 + block_valid_yx[:, 1]
                y_can = canopy_2d[yy, xx]

                # drop NaNs/infs in canopy here so stratification works
                m = np.isfinite(y_can)
                block_valid_yx = block_valid_yx[m]
                yy = yy[m]; xx = xx[m]
                y_can = y_can[m]
                if len(block_valid_yx) == 0:
                    continue

                # assign each valid pixel to a canopy bin
                bin_id = np.digitize(y_can, bins, right=False) - 1  # 0..n_bins-1
                bin_id = np.clip(bin_id, 0, n_bins - 1)

                chosen_local = []

                # sample within each bin
                for b in range(n_bins):
                    idx_b = np.flatnonzero(bin_id == b)
                    if idx_b.size == 0:
                        continue
                    k = min(per_bin, idx_b.size)
                    chosen_local.append(idx_b[rng.choice(idx_b.size, size=k, replace=False)])

                if len(chosen_local) == 0:
                    continue

                chosen_local = np.concatenate(chosen_local)

                # if we didn't reach pixels_per_block (sparse bins), top up uniformly
                remaining = pixels_per_block - chosen_local.size
                if remaining > 0:
                    leftover = np.setdiff1d(np.arange(len(block_valid_yx)), chosen_local, assume_unique=False)
                    if leftover.size > 0:
                        k = min(remaining, leftover.size)
                        chosen_local = np.concatenate([chosen_local,
                                                    leftover[rng.choice(leftover.size, size=k, replace=False)]])

                # convert to flat indices in full grid
                yx = block_valid_yx[chosen_local]
                flat_idx = (y0 + yx[:, 0]) * nx + (x0 + yx[:, 1])

                train_indices.append((flat_idx, by, bx))
    
    return train_indices, test_indices